# High-momentum screen — standalone

**Nothing to clone.** Paste these cells into a blank Colab and run them top to
bottom. The repository is private, so a `git clone` here would ask for a token
and fail; this notebook carries its own code instead.

The rule:

| Leg | Threshold |
|---|---|
| close | **> $5** |
| share volume | **> 300,000/day** |
| quarterly gain (63 sessions) | **> 28%** |

Whatever clears all three is ranked by RS Rank on the 1-year return, tie-broken
by distance below the 52-week high, and the top `N_HOLD` are the picks.

### How faithful this is

Exactly faithful, and that is checkable rather than a claim. This screen has
**no hysteresis band** — `exit_rank` is 0, the rule is "re-screen from scratch,
no memory" — so one day in isolation is the whole computation, and the cells
below reproduce the repository's `finviz_momentum_screen` output for the latest
bar name for name.

(The momentum *ranker* notebook cannot say that: its book carries a band, which
needs yesterday's holdings.)

What is **not** reproduced is history. This evaluates the last bar only, so
there is no equity curve and no turnover here — for those, run the repository.


## 1 · Setup


In [ ]:
# No clone, no token, no repo. `finvizfinance` is only needed for the US
# universe in §2; the Nasdaq-100 path uses Wikipedia and yfinance alone.
!pip install -q "yfinance>=0.2.40" "lxml>=4.9" "finvizfinance>=1.0"

import pandas as pd, numpy as np, yfinance as yf

pd.set_option("display.width", 200)
pd.set_option("display.max_rows", 200)
print("pandas", pd.__version__, "· yfinance", yf.__version__)


## 2 · Choose the universe

**The decision that matters here.**

| | `"ndx"` | `"us"` |
|---|---|---|
| Names | ~100 Nasdaq-100 | ~2,600 US common + ADR |
| Time | under a minute | **several minutes** |
| Source | Wikipedia + yfinance | Finviz screener + yfinance |
| A top-20 is… | usually *everyone who qualified* | a real selection |

On the Nasdaq-100 only a handful of names clear the filter, so asking for 20
mostly returns the whole qualifying set rather than the strongest 20 of a wide
field. Not a bug — it is what an absolute bar does to a 100-name universe — but
the ranking is then barely doing any work.


In [ ]:
UNIVERSE = "ndx"         # "ndx" (fast) or "us" (the real thing, minutes)
N_HOLD   = 20            # how many names to rank down to
START    = "2022-01-01"  # enough history for the 252-day RS lookback

# The live defaults, written out.
MIN_PRICE          = 5.0
MIN_VOLUME         = 300_000.0   # SHARES a day, not dollars
MIN_QUARTER_RETURN = 0.28
QUARTER_LOOKBACK   = 63          # sessions, not calendar months
RS_LOOKBACK        = 252
HIGH_WINDOW        = 252
MIN_HISTORY        = 252
RS_BUCKETS         = 100

print(f"close > ${MIN_PRICE:,.0f} · volume > {MIN_VOLUME:,.0f} shares/day · "
      f"quarterly gain > {MIN_QUARTER_RETURN:.0%} over {QUARTER_LOOKBACK} sessions")
print(f"then RS-ranked on {RS_LOOKBACK} bars, top {N_HOLD} kept")


## 3 · Fetch test 1 — the universe

Either path **raises** rather than falling back to a hardcoded list: a screen
that quietly used stale membership is worse than one that stopped and said so.


In [ ]:
def normalise(symbols):
    """Provider spellings turned into the one yfinance wants.

    Two substitutions, both to a dash, and each has cost a download:
    `.` is a share class (BRK.B -> BRK-B) and `/` is a preferred series
    (ORCL/PD -> ORCL-PD). Left alone the second 404s as "possibly delisted;
    no timezone found", which reads like a dead company rather than a
    misspelled symbol.
    """
    return (pd.Series(list(symbols), dtype="object").astype(str).str.strip()
            .str.upper()
            .str.replace(".", "-", regex=False)
            .str.replace("/", "-", regex=False))


def fetch_ndx():
    """The current Nasdaq-100 constituents, scraped from Wikipedia."""
    for t in pd.read_html("https://en.wikipedia.org/wiki/Nasdaq-100"):
        cols = {str(c).strip().lower() for c in t.columns}
        for key in ("ticker", "symbol"):
            if key in cols:
                col = [c for c in t.columns if str(c).strip().lower() == key][0]
                syms = normalise(t[col])
                syms = [s for s in syms if s.isascii() and 1 <= len(s) <= 6]
                if len(syms) >= 90:
                    return sorted(set(syms))
    raise RuntimeError("no constituents table with >=90 tickers — page changed")


def fetch_us(sleep_sec: int = 1):
    """Every liquid US common stock and ADR, from the Finviz screener.

    Paginates at 20 rows a page, so ~2,600 names is ~120 requests with a pause
    between them. Minutes, and it rate-limits if you hurry it.
    """
    from finvizfinance.screener.overview import Overview

    view = Overview()
    view.set_filter(filters_dict={"Industry": "Stocks only (ex-Funds)",
                                  "Price": "Over $5",
                                  "Average Volume": "Over 300K"})
    df = view.screener_view(order="Ticker", verbose=0, sleep_sec=sleep_sec)
    if df is None or df.empty or "Ticker" not in df.columns:
        raise RuntimeError(f"screener returned nothing usable: "
                           f"{list(df.columns)[:8] if df is not None else None}")
    # `/` matters here: the screener returns preferred series as ORCL/PD and
    # Yahoo wants ORCL-PD. There is one per issuer, so they arrive in batches.
    return sorted(set(normalise(df["Ticker"])))


try:
    TICKERS = fetch_us() if UNIVERSE == "us" else fetch_ndx()
    print(f"FETCH 1 OK — {len(TICKERS)} names in the '{UNIVERSE}' universe")
    print("  ", ", ".join(TICKERS[:12]), "...")
except Exception as exc:
    TICKERS = []
    print(f"FETCH 1 FAILED — {type(exc).__name__}: {exc}")
    print("   Set TICKERS = ['AAPL', ...] yourself and re-run from here.")

# Today's membership applied to all history is SURVIVORSHIP BIAS: the names
# that dropped out are missing, and they are the ones that did badly.


## 4 · Fetch test 2 — prices *and volumes*

Both, because the 300k-share leg is part of the definition. The screen below
**refuses to run** without volume rather than skipping that leg silently — a
screen that ran is a screen that applied its whole rule.


In [ ]:
def fetch_bars(tickers, start=START, batch=100):
    """`(closes, volumes, missing)`. Batched; a failed batch is skipped, not fatal."""
    cl, vl, missing = [], [], []
    for i in range(0, len(tickers), batch):
        chunk = tickers[i:i + batch]
        raw = yf.download(chunk, start=start, auto_adjust=True, progress=False,
                          actions=False, group_by="column", threads=True)
        if raw is None or raw.empty:
            missing += chunk
            continue
        if isinstance(raw.columns, pd.MultiIndex):     # several tickers
            cl.append(raw["Close"])
            vl.append(raw["Volume"])
        else:                                          # exactly one
            cl.append(raw[["Close"]].set_axis(chunk[:1], axis=1))
            vl.append(raw[["Volume"]].set_axis(chunk[:1], axis=1))
        print(f"   ...{min(i + batch, len(tickers))}/{len(tickers)}", end="\r")

    if not cl:
        return pd.DataFrame(), pd.DataFrame(), list(tickers)
    closes = pd.concat(cl, axis=1).sort_index()
    volumes = pd.concat(vl, axis=1).sort_index()
    for f in (closes, volumes):
        f.index = pd.to_datetime(f.index).tz_localize(None).normalize()
    empty = [c for c in closes.columns if closes[c].notna().sum() == 0]
    closes = closes.drop(columns=empty)
    volumes = volumes.reindex(columns=closes.columns)
    return closes, volumes, sorted(set(missing) | set(empty))


closes, volumes, missing = fetch_bars(TICKERS)
safe, _, _ = fetch_bars(["BOXX"])     # the cash leg, fetched separately

if closes.empty:
    print("FETCH 2 FAILED — nothing came back at all.")
else:
    last = closes.index.max()
    on_last = int(closes.loc[last].notna().sum())
    print(f"FETCH 2 OK — {closes.shape[1]} tickers x {len(closes)} sessions")
    print(f"   last bar {last:%Y-%m-%d}, carried by {on_last} of {closes.shape[1]}")
    if missing:
        print(f"   no data for {len(missing)}: {', '.join(missing[:10])}")
    # A last bar only a handful of names carry is the provider mid-publish, not
    # a session. Screening it would screen those few and call it the universe.
    if on_last < 0.5 * closes.shape[1]:
        print(f"   !! that bar is TORN ({on_last} names) — dropping it")
        closes, volumes = closes.iloc[:-1], volumes.iloc[:-1]
    if volumes.isna().all().all():
        print("   !! no volume came back — the 300k leg cannot run")


## 5 · The screen

Three legs, then the ranking. The legs are computed on the last bar; the RS
measure and the 52-week high need the whole window, which is why §2 asks for
more history than the quarter.


In [ ]:
def rs_rank_order(candidates, pct_off_high, buckets=RS_BUCKETS):
    """RS Rank ordering: bucket, sort descending, tie-break on the high.

    `rank(method='first')` makes the input strictly increasing so `qcut` splits
    it into equal buckets without collapsing duplicate edges.
    """
    if candidates.empty:
        return candidates, candidates
    q = min(buckets, len(candidates))
    if q < 2:
        rs = pd.Series(1.0, index=candidates.index)
    else:
        rs = pd.Series(pd.qcut(candidates.rank(method="first"), q=q,
                               labels=False, duplicates="drop"),
                       index=candidates.index).astype(float) + 1.0
    frame = pd.DataFrame({
        "rs": rs,
        "off_high": pct_off_high.reindex(candidates.index).fillna(np.inf),
    }).sort_values(["rs", "off_high"], ascending=[False, True])
    return candidates.reindex(frame.index), rs.reindex(frame.index)


def run_screen(closes, volumes, n_hold=N_HOLD):
    """The screen on the last bar. `(picks, rs, legs, measures, asof)`."""
    if closes.empty:
        raise RuntimeError("no prices — check the fetch tests above")
    if volumes is None or volumes.isna().all().all():
        raise ValueError("the 300k-share leg needs volumes; it is part of the "
                         "definition and is not skipped silently")
    px = closes.sort_index()
    vol = volumes.reindex(index=px.index, columns=px.columns).ffill()
    asof = px.index.max()

    quarter = (px / px.shift(QUARTER_LOOKBACK) - 1.0).loc[asof]
    perf_1y = (px / px.shift(RS_LOOKBACK) - 1.0).loc[asof]
    high_52w = px.rolling(HIGH_WINDOW, min_periods=HIGH_WINDOW).max()
    off_high = (1.0 - px / high_52w).loc[asof]
    history = px.notna().cumsum().loc[asof]

    legs = {
        f"close > ${MIN_PRICE:,.0f}": px.loc[asof] > MIN_PRICE,
        f"volume > {MIN_VOLUME:,.0f}": vol.loc[asof] > MIN_VOLUME,
        f"quarter > {MIN_QUARTER_RETURN:.0%}": quarter > MIN_QUARTER_RETURN,
    }
    ok = np.logical_and.reduce(list(legs.values()))
    ok = pd.Series(ok, index=px.columns) & (history >= MIN_HISTORY) & perf_1y.notna()

    cand = perf_1y[ok].dropna()
    order, rs = rs_rank_order(cand, off_high)
    # No hysteresis band on this screen: re-screen from scratch, no memory.
    picks = list(order.index) if n_hold == 0 else list(order.index[:n_hold])
    measures = pd.DataFrame({"Close": px.loc[asof], "Volume": vol.loc[asof],
                             "Quarter": quarter, "1-year": perf_1y,
                             "Off high": off_high})
    return picks, rs, legs, measures, asof


picks, rs, legs, measures, asof = run_screen(closes, volumes)
print(f"{asof:%Y-%m-%d} — {len(picks)} of {N_HOLD} slots filled")
if len(picks) < N_HOLD:
    print("   fewer names cleared the filter than the screen ranks down to, so "
          "this is everyone who qualified rather than the strongest N.")
print()
print(", ".join(picks) if picks else "nothing qualified — the book is in cash")


## 6 · The picks, with the numbers that chose them


In [ ]:
if picks:
    table = measures.loc[picks].copy()
    table.insert(0, "RS", rs.reindex(picks).round(0))
    table.insert(0, "Rank", range(1, len(table) + 1))
    table.insert(3, "$ volume", table["Close"] * table["Volume"])
    display(table.style.format({
        "Close": "${:,.2f}", "Volume": "{:,.0f}", "$ volume": "${:,.0f}",
        "Quarter": "{:+.1%}", "1-year": "{:+.1%}", "Off high": "{:.1%}",
        "RS": "{:.0f}"})
        .background_gradient(subset=["Quarter", "1-year"], cmap="Greens"))
    print("Rank is the screen's own order: RS Rank on the 1-year return, "
          "tie-broken by the smallest distance below the 52-week high.")
else:
    table = pd.DataFrame()
    print("Nothing to show.")


## 7 · Where everyone else was lost

The funnel is the interesting half. One leg usually does nearly all the work,
and which one tells you what kind of market this is.


In [ ]:
rows = [{"Leg": k, "Pass": int(v.sum()), "Share": v.sum() / len(v)}
        for k, v in legs.items()]
all_three = np.logical_and.reduce(list(legs.values()))
rows.append({"Leg": "ALL THREE", "Pass": int(all_three.sum()),
             "Share": all_three.sum() / len(all_three)})
display(pd.DataFrame(rows).style.format({"Share": "{:.1%}"}).hide(axis="index"))

qualified = sorted(closes.columns[all_three])
cut = [t for t in qualified if t not in picks]
print(f"{len(qualified)} qualified, {len(picks)} kept.")
print("  cut by the ranking:",
      ", ".join(cut) if cut else "none — every qualifying name fitted")


## 8 · Take it with you


In [ ]:
if not table.empty:
    out = table.copy()
    out.index.name = "Ticker"
    fname = f"high_momentum_{UNIVERSE}_{asof:%Y%m%d}.csv"
    out.to_csv(fname, encoding="utf-8-sig")
    print("wrote", fname)
    try:                       # Colab only; a no-op anywhere else
        from google.colab import files
        files.download(fname)
    except Exception:
        print("(not on Colab — the file is in the working directory)")


---

Every figure above comes from data fetched in this notebook, and the screen is
the live rule: same three legs, same thresholds, same RS ordering, same
no-memory selection. What is missing is history — this is the last bar only, so
no equity curve and no turnover. The universe is also today's membership, which
is fine for "what is strong now" and is survivorship bias in a backtest.
